In [ ]:
import torch
import torch.nn as nn

x = torch.tensor([[1],[2],[3],[4],[5],[6],[7],[8]], dtype = torch.float32)
y = torch.tensor([[2],[4],[6],[8],[10],[12],[14],[16]],  dtype = torch.float32)

n_samples, n_features = x.shape

print (f"The number of samples be {n_samples}, with features {n_features}")





In [ ]:
# define moel

class LinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self,x):
      return self.linear(x)

input_size , output_size = n_features, n_features
x_test = torch.tensor([12], dtype = torch.float32)
model = LinearRegression(input_size, output_size)
print(f"Prediction before training : f({x_test.item()}) = {model(x_test).item():.3f}")


learning_rate = 0.01
epochs = 1000

loss = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

for epoch in range(epochs):
  y_pred = model(x)

  l = loss(y, y_pred)

  l.backward()

  optimizer.step()

  optimizer.zero_grad()

  if (epoch+1)%10 == 0:
    w,b = model.parameters()
    print(f"epoch {epoch+1}, w = {w[0][0].item():.3f}, b = {b[0].item():.3f}")



In [ ]:
 print(model(x_test))
 print(f"{x_test.item()} = {model(x_test).item():.3f}")

# **ANN**

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = 784 # 28*28
hidden_size = 500
num_classes = 10
num_epochs = 5
batch_size = 100
learning_rate = 0.001

print(device)

In [ ]:
train_dataset = torchvision.datasets.MNIST(root = "./data",
                                           train = True ,
                                           download = True,
                                           transform = transforms.ToTensor())
test_dataset = torchvision.datasets.MNIST(root = "./data",
                                          train = False,
                                          transform = transforms.ToTensor(),
                                          download = True)

In [ ]:
train_loader = torch.utils.data.DataLoader(dataset = train_dataset,
                                           batch_size = batch_size,
                                           shuffle = True
                                           )

test_loader = torch.utils.data.DataLoader(dataset = test_dataset,
                                          batch_size = batch_size,
                                          shuffle = True
                                          )

In [ ]:
examples = iter(train_loader)
example_data, example_targets =next(examples)
print(example_data.shape)

In [ ]:
for i in range(6):
  plt.subplot(2,3,i+1)
  plt.imshow(example_data[i][0], cmap = "gray")

plt.show()

In [ ]:
class NeuralNet(nn.Module):
  def __init__(self, input_size, hidden_size, num_classes):
    super(NeuralNet, self).__init__()
    self.l1 = nn.Linear(input_size, hidden_size)
    self.relu = nn.ReLU()
    self.l2 = nn.Linear(hidden_size, num_classes)

  def forward(self, x):
    out = self.l1(x)
    out = self.relu(out)
    out = self.l2(out)
    return out

In [ ]:
print(len(train_loader))

In [ ]:
model = NeuralNet(input_size, hidden_size, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer =  torch.optim.Adam(model.parameters(), lr = learning_rate)

n_total_steps = len(train_loader)

for epoch in range(num_epochs):
  for i, (images, labels) in enumerate(train_loader):
    images = images.reshape(-1, 28*28).to(device)
    labels = labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (i+1)%100 == 0:
      print(f"epoch {epoch+1}/{num_epochs}, step {i+1}/{n_total_steps}, loss = {loss.item():.4f}")

In [ ]:
with torch.no_grad():
  n_correct = 0
  n_samples = len(test_loader.dataset)

  for images, labels in test_loader:
    images = images.reshape(-1, 28*28).to(device)
    labels = labels.to(device)


    outputs = model(images)

    _, predictions = torch.max(outputs, 1)
    n_correct += (predictions == labels).sum().item()

    acc = 100.0 * n_correct / n_samples
  print(f"Accuracy of the model on the {n_samples} test images: {acc}")

# **CNN**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 10
batch_size = 32
learning_rate = 0.001

# the composed transformation first converts an image to a tensor and then normalizes the pixel values so that they have a mean 0 and SD 1
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# we will use CIFAR10 data set 6000 32*32 colour images 10 classes with 6000 images per class

train_dataset = torchvision.datasets.CIFAR10(root="./data",
                                             train = True,
                                             download = True,
                                             transform = transform)

test_dataset = torchvision.datasets.CIFAR10(root="./data",
                                            download = True,
                                            train = False ,
                                            transform = transform)

train_loader = torch.utils.data.DataLoader(dataset = train_dataset,
                                           batch_size = batch_size,
                                           shuffle = True)

test_loader = torch.utils.data.DataLoader(dataset = test_dataset,
                                          batch_size = batch_size,
                                          shuffle = True)

classes = ("plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")

def imshow(img):
  img = img/2 + 0.5
  npimg = img.numpy()
  plt.imshow(np.transpose(npimg, (1,2,0)))
  plt.show()

detailer = iter(train_loader)
images, labels = next(detailer)
img_grid = torchvision.utils.make_grid(images[0:25], nrow=5)
imshow(img_grid)

In [ ]:
class ConvNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size = 3)
    self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)
    self.Conv2 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3)
    self.conv3 = nn.Conv2d(in_channels = 64, out_channels = 64, kernel_size = 3)
    self.fc1 = nn.Linear(64*4*4, 64)
    self.fc2 = nn.Linear(64, 10)

  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = self.pool(x)
    x = F.relu(self.Conv2(x))
    x = self.pool(x)
    x = F.relu(self.conv3(x))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x


In [ ]:
model = ConvNet().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

n_total_steps = len(train_loader)
for epoch in range(epochs):
  running_loss = 0.0
  for i, (images, labels) in enumerate(train_loader):
    images = images.to(device)
    labels = labels.to(device)

    output = model(images)
    loss = criterion(output, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step

    running_loss = running_loss + loss.item()

  print(f"[{epoch+1}], loss = {running_loss/n_total_steps :.3f}")

PATH = "./cnn.pth"
torch.save(model.state_dict(), PATH)


In [ ]:
loaded_model = ConvNet()
loaded_model.load_state_dict(torch.load(PATH))
loaded_model.to(device)
loaded_model.eval()

with torch.no_grad():
  n_correct = 0
  n_samples = len(test_loader.dataset)

  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)


    outputs = model(images)
    _, predictions = torch.max(outputs, 1)
    n_correct += (predictions == labels).sum().item()


  acc = 100.0 * n_correct / n_samples
  print(f"Accuracy of the model on the {n_samples} test images: {acc} %")